# Workspace — News to Telegram

Every morning something happens in technology and you find out three days later. In this
project you build a **digest agent**: it reads the day's stories, decides which three
matter, and sends them to your phone.

This workspace is open-ended. The milestones below are the contract, the route is yours.
Everything runs offline on the `FakeLLM` and a recorded fixture, so the whole pipeline
works with no key and no network. The only part that needs a secret is the delivery, and
it skips politely without one.

**Where the idea came from.** The Hacker News newsletter agent in the
[Hands-On AI Engineering](https://github.com/Sumanth077/Hands-On-AI-Engineering)
collection, which is worth browsing for what people build with agents. Ours is written
from scratch on the course's own rails and differs in three ways that are the lesson:
the model call goes behind the `LLMClient` seam so it runs deterministically in CI, the
delivery is Telegram rather than Gmail SMTP so nobody needs 2-Step Verification and an
app password, and the whole thing is checked against a recorded fixture so a refactor
that breaks it fails loudly.

**Before you depend on any repository**, including that one, spend two minutes finding
out what you are taking on:

```bash
git grep -l "^<<<<<<< " HEAD | wc -l      # does the default branch even compile
python -m compileall -q . 2>&1 | head     # the same question, louder
ls LICENSE                                # is there a grant, or only a badge
grep -ci licen CONTRIBUTING.md            # did contributors agree to anything
git log --format='%an' | sort | uniq -c   # who actually maintains it
grep -rhoE "[A-Z_]+_API_KEY" . | sort -u  # what will it cost you to run
```

Run those before you build on someone's work. The answers are often surprising, and
they are always cheaper to learn now than after you have committed to it.

In [ ]:
# Setup — the course environment has everything this workspace needs.
# (From a fresh clone: `uv sync --group dev` at the repo root.)
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

WORKSPACE = REPO_ROOT / "workspaces" / "news-to-telegram"
FIXTURE = WORKSPACE / "fixtures" / "hn-front-page.json"
print(f"repo root: {REPO_ROOT}")
print(f"fixture:   {FIXTURE.relative_to(REPO_ROOT)}")

## Chapter 1 — The source

Hacker News is searchable through Algolia with no key and no account, which makes it a
rare thing: a real API you can teach against for free. We take the front page, because
the newest stories have no votes yet and there is nothing to rank.

**Milestone:** stories on screen, from the fixture, with the live path one argument away.
The fixture states what it is and is not evidence of; open it and read that block.

In [ ]:
def load_stories(live: bool = False) -> list[dict]:
    """The ten stories on the Hacker News front page.

    Recorded by default so this notebook runs with no network. `live=True` calls
    the Algolia search API, which needs no key and no account.
    """
    if not live:
        return json.loads(FIXTURE.read_text())["hits"]

    import urllib.request

    url = "https://hn.algolia.com/api/v1/search?tags=front_page&hitsPerPage=10"
    with urllib.request.urlopen(url, timeout=20) as response:  # noqa: S310 - pinned https host
        payload = json.load(response)
    return [
        {
            "objectID": hit["objectID"],
            "title": hit.get("title"),
            "url": hit.get("url") or f"https://news.ycombinator.com/item?id={hit['objectID']}",
            "points": hit.get("points"),
            "num_comments": hit.get("num_comments"),
            "created_at": hit.get("created_at"),
            "author": hit.get("author"),
        }
        for hit in payload.get("hits", [])
    ]


stories = load_stories()
print(f"{len(stories)} stories, recorded {json.loads(FIXTURE.read_text())['_provenance']['recorded']}\n")
for story in stories[:3]:
    print(f"  {story['points']:>4} pts  {story['title'][:64]}")

## Chapter 2 — The judgement

Ranking is the only part a model is actually needed for, so it is the only part that goes
through the seam. `LLMClient.complete(system, user)` is one method, which is what makes it
swappable: `FakeLLM` here, a real provider behind `BOOTCAMP_PROVIDER` when you want one.

**Milestone:** three stories chosen and a sentence saying why. Note that the editor is
allowed to pick none, and that the prompt says so. An agent that must always find three
significant stories will invent significance on a slow day.

In [ ]:
from bootcamp_agent.llm import FakeLLM

SYSTEM = (
    "You are a technology news editor. Given a numbered list of stories, reply with the "
    "numbers of the three that matter most, most significant first, as JSON: "
    '{"picks": [1, 2, 3], "why": "one sentence"}. '
    "If none of them are significant, reply {\"picks\": [], \"why\": \"...\"}."
)


def as_prompt(stories: list[dict]) -> str:
    lines = [f"{i}. {s['title']} ({s['points']} points)" for i, s in enumerate(stories, 1)]
    return "Stories:\n" + "\n".join(lines)


# FakeLLM keyword-matches the user prompt, so the notebook is deterministic in CI.
# A real model is one environment variable away: see SETUP.md, BOOTCAMP_PROVIDER.
editor = FakeLLM(
    responses={"Stories:": '{"picks": [1, 2, 3], "why": "the three the front page is talking about"}'},
    default='{"picks": [], "why": "no canned answer; a real model would decide here"}',
)

verdict = json.loads(editor.complete(system=SYSTEM, user=as_prompt(stories)))
picked = [stories[i - 1] for i in verdict["picks"]]
print(f"picked {len(picked)}: {verdict['why']}\n")
for story in picked:
    print(f"  {story['title'][:70]}")

## Chapter 3 — The delivery

One `POST` to `api.telegram.org`. No SMTP, no SSL dance, no app password.

To make it real: message `@BotFather` on Telegram, send `/newbot`, and it hands you a
token. Message your new bot once, then read your chat id from
`https://api.telegram.org/bot<token>/getUpdates`. Put both in your environment, never in
this notebook.

**Milestone:** a digest that reads well on a phone, and a send that skips rather than
crashes when the token is missing.

In [ ]:
import os
import urllib.parse
import urllib.request


def build_digest(picked: list[dict], why: str) -> str:
    """The message itself. Plain text, because it has to read well on a phone."""
    lines = ["*Today in tech*", ""]
    for story in picked:
        lines.append(f"• [{story['title']}]({story['url']})  _{story['points']} points, {story['num_comments']} comments_")
    lines += ["", f"_{why}_"]
    return "\n".join(lines)


def send_to_telegram(text: str) -> str:
    """Deliver the digest, or say why it was skipped. Never raises.

    Set TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID to make this real. Without them
    the notebook still runs, which is the point: the pipeline is testable and
    the delivery is the only part that needs a secret.
    """
    token = os.environ.get("TELEGRAM_BOT_TOKEN", "").strip()
    chat_id = os.environ.get("TELEGRAM_CHAT_ID", "").strip()
    if not token or not chat_id:
        return "skipped: set TELEGRAM_BOT_TOKEN and TELEGRAM_CHAT_ID to deliver for real"

    body = urllib.parse.urlencode(
        {"chat_id": chat_id, "text": text, "parse_mode": "Markdown"}
    ).encode()
    request = urllib.request.Request(  # noqa: S310 - pinned https host
        f"https://api.telegram.org/bot{token}/sendMessage", data=body
    )
    try:
        with urllib.request.urlopen(request, timeout=20) as response:
            return f"delivered: {json.load(response).get('ok')}"
    except Exception as error:  # noqa: BLE001 - a failed send must not kill the run
        return f"send failed: {type(error).__name__}"


digest = build_digest(picked, verdict["why"])
print(digest)
print()
print(send_to_telegram(digest))

## Chapter 4 — The proof

A pipeline you cannot test is a pipeline you will not change. These two checks are the
smallest honest pair: the digest still gets built from the fixture, and the delivery still
declines to explode without a secret.

**Milestone:** both green, and a third check of your own for whatever you build next.

In [ ]:
def test_digest_is_built_from_the_fixture() -> None:
    """The check that survives a refactor: three stories in, three links out."""
    stories = load_stories()
    assert len(stories) == 10, "the fixture holds ten stories"

    verdict = json.loads(editor.complete(system=SYSTEM, user=as_prompt(stories)))
    picked = [stories[i - 1] for i in verdict["picks"]]
    digest = build_digest(picked, verdict["why"])

    assert len(picked) == 3, "the editor picks three"
    for story in picked:
        assert story["url"] in digest, f"{story['title'][:40]} lost its link"
    assert digest.startswith("*Today in tech*")


def test_delivery_skips_without_a_token() -> None:
    """No secret, no send, no crash."""
    saved = os.environ.pop("TELEGRAM_BOT_TOKEN", None)
    try:
        assert send_to_telegram("hello").startswith("skipped:")
    finally:
        if saved is not None:
            os.environ["TELEGRAM_BOT_TOKEN"] = saved


test_digest_is_built_from_the_fixture()
test_delivery_skips_without_a_token()
print("✅ both checks passed")

## Start coding here

Use as many cells as you need. Things worth trying, in rough order of difficulty:

- Filter by score or age before the model sees anything. Cheaper and often better.
- Fetch each story's page and summarise the content rather than the headline.
- Run it on a schedule with cron, and make it remember what it already sent.
- Swap the source: an RSS feed, a subreddit, your own team's changelog.

In [ ]:
# Start coding here
# Use as many cells as you need.